In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.mistralai import MistralAI

import chromadb
from chromadb.config import Settings
from dotenv import load_dotenv
from IPython.display import Markdown, display
import os
import textwrap


In [ ]:
# Mengambil API Key
load_dotenv()

mistral_api_key = os.getenv("MISTRAL_API_KEY")
chroma_api_key = os.getenv("CHROMA_API_KEY")
chroma_tenant = os.getenv("CHROMA_TENANT")
chroma_database = os.getenv("CHROMA_DATABASE")

if not mistral_api_key or not chroma_api_key:
    raise ValueError("MISTRAL_API_KEY dan CHROMA_API_KEY Salah")

# Setup Mistral & Chroma client
mistral_llm = MistralAI(api_key=mistral_api_key, model="mistral-large-latest")
client = chromadb.CloudClient(
    api_key=chroma_api_key,
    tenant=chroma_tenant,
    database=chroma_database
)

print("Koneksi ke Mistral & ChromaDB berhasil.")


Koneksi ke Mistral & ChromaDB berhasil.


In [ ]:
# Membuat Divisi atau Collection Baru
def create_new_collection(collection_name: str):
    """Membuat Divisi baru."""
    try:
        existing = [col.name for col in client.list_collections()]
        if collection_name in existing:
            print(f"Divisi '{collection_name}' sudah ada.")
            return

        client.create_collection(name=collection_name)
        print(f"Divisi '{collection_name}' berhasil dibuat.")
    except Exception as e:
        print(f"Gagal membuat Divisi: {str(e)}")


# Input interaktif
collection_name = input("Masukkan nama Divisi baru: ")


create_new_collection(collection_name)

Collection 'Halo' berhasil dibuat.


In [ ]:
# Setup Embedding
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")


In [38]:
# Memilih Divisi yang diinginkan
collections = client.list_collections()

if not collections:
    print("Belum ada Divisi")
else:
    print("\nDaftar divisi tersedia:")
    for i, c in enumerate(collections):
        print(f"{i+1}. {c.name}")

    choice = int(input("\nPilih nomor divisi yang ingin digunakan: ")) - 1
    selected_collection = collections[choice].name
    chroma_collection = client.get_collection(selected_collection)
    print(f"Divisi '{selected_collection}' dipilih.")



Daftar divisi tersedia:
1. HCM
2. MRO
3. PM_K3LH
4. baru
Divisi 'baru' dipilih.


In [40]:
# Upload dokumen baru ke collection
documents = SimpleDirectoryReader("./data/").load_data()
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context, embed_model=embed_model
)

In [41]:
# Memuat index dari collection yang sudah dipilih
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_vector_store(
    vector_store, storage_context=storage_context, embed_model=embed_model
)

In [42]:
# Melakukan Prompt
query_engine = index.as_query_engine(llm=mistral_llm)
response = query_engine.query("Apa komitmen perusahaan terhadap K3LH?")

display(Markdown(textwrap.fill(str(response), width=100)))


Informasi mengenai komitmen perusahaan terhadap **Kesehatan, Keselamatan Kerja, dan Lingkungan Hidup
(K3LH)** tidak tercantum dalam data yang tersedia. Untuk pertanyaan ini, disarankan menghubungi
**Divisi Pemasaran & Penjualan** atau **Div MRO** melalui email **defense@pindad.com** atau
**sales@pindad.com** guna mendapatkan jawaban yang akurat.

In [ ]:
# Menghapus Data dalam Collection atau Divisi
collections = client.list_collections()

if collections:
    for i, c in enumerate(collections):
        print(f"{i+1}. {c.name}")

    choice = int(input("\nPilih nomor Divisi yang ingin DIKOSONGKAN: ")) - 1
    selected_collection = collections[choice].name
    chroma_collection = client.get_collection(selected_collection)

    confirm = input(f"Apakah yakin ingin menghapus SEMUA data di divisi '{selected_collection}'? (y/n): ")
    if confirm.lower() == "y":
        # Ambil semua ID dokumen yang ada di collection
        all_items = chroma_collection.get()  # Tidak perlu include
        if all_items["ids"]:
            chroma_collection.delete(ids=all_items["ids"])
        else:
            print("Data pada divisi sudah kosong.")
else:
    print("Belum ada cdivisi.")


1. HCM
2. MRO
3. PM_K3LH
4. baru
5. Halo


In [51]:
# Ambil daftar collection dari Chroma
collections = client.list_collections()

if collections:
    print("Daftar divisi yang tersedia:\n")
    for i, c in enumerate(collections):
        print(f"{i+1}. {c.name}")

    choice = int(input("\nPilih nomor divisi yang ingin DIHAPUS: ")) - 1
    selected_collection = collections[choice].name

    confirm = input(f"Yakin ingin MENGHAPUS divisi '{selected_collection}'? (y/n): ")
    if confirm.lower() == "y":
        client.delete_collection(selected_collection)
        print(f"Divisi '{selected_collection}' telah dihapus.")
    else:
        print("Penghapusan dibatalkan.")
else:
    print("Divisi tidak ada.")


Daftar divisi yang tersedia:

1. HCM
2. MRO
3. PM_K3LH
4. baru
5. Halo
Divisi 'Halo' telah dihapus.
